In [1]:
'''
task: FOL classification based on NL text
model: FOLIO NL t5-large
dataset: SemEval-2026
evaluation: supervised fine-tuning
'''
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

# load train, val and test splits
train = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/SemEval2026/data/train.csv")
val = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/SemEval2026/data/val.csv")
test = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/SemEval2026/data/test.csv")

# transform data to acceptable format for model training
train_df = pd.DataFrame(train)
val_df = pd.DataFrame(val)
test_df = pd.DataFrame(test)

train_df = train_df.applymap(str)
val_df = val_df.applymap(str)
test_df = test_df.applymap(str)

/tmp/ipython-input-1243017007.py:13: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  train_df = train_df.applymap(str)
/tmp/ipython-input-1243017007.py:14: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  val_df = val_df.applymap(str)
/tmp/ipython-input-1243017007.py:15: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  test_df = test_df.applymap(str)


In [3]:
# start preparing for QA pipeline
!pip install datasets
! pip install -U accelerate
! pip install -U transformers
!pip install transformers
!pip install evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 134.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.3
    Uninstalling transformers-4.57.3:
      Successfully uninstalled transformers-4.57.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.9 MB/s eta 0:00:00


In [4]:
import torch
import json
from tqdm import tqdm
import torch.nn as nn
from torch.optim import Adam
import nltk
import spacy
import string
import evaluate  # Bleu
from torch.utils.data import Dataset, DataLoader, RandomSampler
import pandas as pd
import numpy as np
import transformers
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from transformers import T5Tokenizer, T5Model, T5ForConditionalGeneration, T5TokenizerFast

import warnings
warnings.filterwarnings("ignore")

In [5]:
MODEL = T5ForConditionalGeneration.from_pretrained("/content/drive/MyDrive/Colab Notebooks/folio/models/flan-t5-large_logic_nl_model")
TOKENIZER = T5TokenizerFast.from_pretrained("/content/drive/MyDrive/Colab Notebooks/folio/models/flan-t5-large_logic_nl_tokenizer")

In [6]:
# set model parameters
Q_LEN = 512   # Question Length
T_LEN = 512    # Target Length
BATCH_SIZE = 4
DEVICE = "cuda:0"
OPTIMIZER = Adam(MODEL.parameters(), lr=0.00001)

In [7]:
class QA_Dataset(Dataset):
    def __init__(self, tokenizer, dataframe, q_len, t_len):
        self.tokenizer = tokenizer
        self.q_len = q_len
        self.t_len = t_len
        self.data = dataframe
        self.subject = self.data["syllogism"]
        self.relation = self.data['validity']

    def __len__(self):
        return len(self.subject)

    def __getitem__(self, idx):
        subject = self.subject[idx]
        relation = self.relation[idx]

        subject_tokenized = self.tokenizer(subject, max_length=self.q_len, padding="max_length",
                                                    truncation=True, add_special_tokens=True)
        relation_tokenized = self.tokenizer(relation, max_length=self.t_len, padding="max_length",
                                          truncation=True, add_special_tokens=True)

        labels = torch.tensor(relation_tokenized["input_ids"], dtype=torch.long)
        labels[labels == 0] = -100

        return {
            "input_ids": torch.tensor(subject_tokenized["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(subject_tokenized["attention_mask"], dtype=torch.long),
            "labels": labels,
            "decoder_attention_mask": torch.tensor(relation_tokenized["attention_mask"], dtype=torch.long)
        }

In [8]:
train_dataset = QA_Dataset(TOKENIZER, train_df, Q_LEN, T_LEN)
val_dataset = QA_Dataset(TOKENIZER, val_df, Q_LEN, T_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

In [9]:
#torch.cuda.empty_cache()
MODEL.to('cuda')

T5ForConditionalGeneration(
  (shared): Embedding(32128, 1024)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 1024)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1024, out_features=1024, bias=False)
              (k): Linear(in_features=1024, out_features=1024, bias=False)
              (v): Linear(in_features=1024, out_features=1024, bias=False)
              (o): Linear(in_features=1024, out_features=1024, bias=False)
              (relative_attention_bias): Embedding(32, 16)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=1024, out_features=2816, bias=False)
              (wi_1): Linear(in_features=1024, out_features=2816, bias=False)
       

In [10]:
train_loss = 0
val_loss = 0
train_batch_count = 0
val_batch_count = 0

for epoch in range(5):
    MODEL.train()
    for batch in tqdm(train_loader, desc="Training batches"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        decoder_attention_mask = batch["decoder_attention_mask"].to(DEVICE)

        outputs = MODEL(
                          input_ids=input_ids,
                          attention_mask=attention_mask,
                          labels=labels,
                          decoder_attention_mask=decoder_attention_mask
                        )

        OPTIMIZER.zero_grad()
        outputs.loss.backward()
        OPTIMIZER.step()
        train_loss += outputs.loss.item()
        train_batch_count += 1

    #Evaluation
    MODEL.eval()
    for batch in tqdm(val_loader, desc="Validation batches"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        decoder_attention_mask = batch["decoder_attention_mask"].to(DEVICE)

        outputs = MODEL(
                          input_ids=input_ids,
                          attention_mask=attention_mask,
                          labels=labels,
                          decoder_attention_mask=decoder_attention_mask
                        )

        OPTIMIZER.zero_grad()
        outputs.loss.backward()
        OPTIMIZER.step()
        val_loss += outputs.loss.item()
        val_batch_count += 1

    print(f"{epoch+1}/{5} -> Train loss: {train_loss / train_batch_count}\tValidation loss: {val_loss/val_batch_count}")

Validation batches: 100%|██████████| 94/94 [01:16<00:00,  1.22it/s]


1/5 -> Train loss: 0.249722918237936	Validation loss: 0.159554721183203


Validation batches: 100%|██████████| 94/94 [01:16<00:00,  1.23it/s]


2/5 -> Train loss: 0.20305946981038978	Validation loss: 0.11026595851550117


Validation batches: 100%|██████████| 94/94 [01:16<00:00,  1.23it/s]


3/5 -> Train loss: 0.17145783984241017	Validation loss: 0.07884578359088736


Validation batches: 100%|██████████| 94/94 [01:16<00:00,  1.23it/s]


4/5 -> Train loss: 0.1448007620625852	Validation loss: 0.059987797218195256


Validation batches: 100%|██████████| 94/94 [01:16<00:00,  1.22it/s]

5/5 -> Train loss: 0.12723301492360212	Validation loss: 0.048647148066799886


In [11]:
MODEL.save_pretrained("/content/drive/MyDrive/Colab Notebooks/SemEval2026/models/folio_flan-t5-large_semeval_nl_model")
TOKENIZER.save_pretrained("/content/drive/MyDrive/Colab Notebooks/SemEval2026/models/folio_flan-t5-large_semeval_nl_tokenizer")

('/content/drive/MyDrive/Colab Notebooks/SemEval2026/models/folio_flan-t5-large_semeval_nl_tokenizer/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/SemEval2026/models/folio_flan-t5-large_semeval_nl_tokenizer/special_tokens_map.json',
 '/content/drive/MyDrive/Colab Notebooks/SemEval2026/models/folio_flan-t5-large_semeval_nl_tokenizer/spiece.model',
 '/content/drive/MyDrive/Colab Notebooks/SemEval2026/models/folio_flan-t5-large_semeval_nl_tokenizer/added_tokens.json',
 '/content/drive/MyDrive/Colab Notebooks/SemEval2026/models/folio_flan-t5-large_semeval_nl_tokenizer/tokenizer.json')

In [12]:
# evaluation metrics

import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report


def predict_answer(subject, ref_relation=None):
    inputs = TOKENIZER(subject, return_tensors="pt").to(MODEL.device)

    outputs = MODEL.generate(input_ids=inputs["input_ids"], max_new_tokens=10)

    predicted_relation = TOKENIZER.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True)[0]

    print("syllogism: \n", subject)
    print("true label: \n", ref_relation)
    print("predicted label: \n", predicted_relation)

    return predicted_relation

In [13]:
# test predictions
reference_labels = []
predicted_labels = []
for index, row in test_df.iterrows():
  premises = row["syllogism"]
  label = row["validity"]

  predicted_label = predict_answer(premises, label)
  reference_labels.append(label)
  predicted_labels.append(predicted_label)

# calculate classification metrics
print("Accuracy:", accuracy_score(reference_labels, predicted_labels))
print("Precision:", precision_score(reference_labels, predicted_labels, average="macro"))
print("Recall:", recall_score(reference_labels, predicted_labels, average="macro"))
print("F1:", f1_score(reference_labels, predicted_labels, average="macro"))
print("Classification Report:", classification_report(reference_labels, predicted_labels))

syllogism: 
 All forms of bacteria are classified as communities. There are some organs that do not belong to the category of communities. Therefore, some organs are not types of bacteria.
true label: 
 True
predicted label: 
 True
syllogism: 
 Every cat is a mammal. The set of dolphins contains no cats. Every single dolphin is a mammal.
true label: 
 False
predicted label: 
 False
syllogism: 
 Anything that is a tomato is also a planet. There is no planet that is a food. It is thus the case that no food is a tomato.
true label: 
 True
predicted label: 
 True
syllogism: 
 Every single eagle is a bird. Nothing that is a bat is a bird. From this, it can be concluded that no bat is an eagle.
true label: 
 True
predicted label: 
 False
syllogism: 
 Any creature that is a bird is an animal. A portion of animals are kept as pets. Some pets are not birds.
true label: 
 False
predicted label: 
 False
syllogism: 
 Anything which is a cloud is made of rock. Some weather phenomena are known to be